# Galaxy Redshift and Pulsar Detection

AIG100 machine learning project in progress.


# Regression— Predicting Galaxy Redshift

The star_classification.csv dataset contains observations of galaxies, stars, and quasars from the Sloan Digital Sky Survey. For this exercise, we will keep only the galaxies. Each galaxy has five brightness measurements (u, g, r, i, and z) taken through different filters, along with a measured redshift. Redshift describes how much the wavelengths of light from the galaxy have stretched and is related to how far away the galaxy is.

Our goal is to estimate a galaxy’s redshift using only its five brightness measurements. Since redshift is a continuous numerical value, this is a regression problem. We will compare how accurately linear regression and random forest regression predict redshift for galaxies they did not see during training.

https://www.sdss3.org/dr10/imaging/imaging_basics.php

https://www.sdss4.org/dr12/help/glossary/

https://arxiv.org/abs/astro-ph/0006396

Here are the columns in the order they appear in SDSS imaging guide ·

| Column | What it means | Role in this project |
|---|---|---|
| `obj_ID` | Catalog ID for the object in the imaging data. | Leave out |
| `alpha` | Right ascension, one coordinate for locating an object on the sky. Roughly like longitude. | Leave out |
| `delta` | Declination, the other sky coordinate. Roughly like latitude. | Leave out |
| `u` | Apparent brightness measured through the near-ultraviolet filter. | Input |
| `g` | Apparent brightness measured through the blue-green filter. | Input |
| `r` | Apparent brightness measured through the red filter. | Input |
| `i` | Apparent brightness measured through a filter at longer wavelengths than `r`. | Input |
| `z` | Apparent brightness measured through the longest-wavelength filter of these five. | Input |
| `run_ID` | Identifies the imaging scan. | Leave out |
| `rerun_ID` | Identifies a processing version of the imaging data. | Leave out |
| `cam_col` | Identifies the camera column that recorded the image. | Leave out |
| `field_ID` | Identifies a section of the imaging scan. | Leave out |
| `spec_obj_ID` | Catalog ID for the spectrum, a record of the object’s light broken down by wavelength. | Leave out |
| `class` | Object type. The file uses labels such as `GALAXY`, `STAR`, and `QSO` for quasar. | Filter to `GALAXY` |
| `redshift` | Fractional change in the wavelength of identifiable features in the spectrum. | Target |
| `plate` | Identifies the plate used to collect the spectrum. | Leave out |
| `MJD` | Modified Julian Date, a numeric way of recording the observation date. | Leave out |
| `fiber_ID` | Identifies the optical fiber that carried the object’s light to the spectrograph. | Leave out |



A redshift of 0.635 means a feature’s observed wavelength is about 63.5% longer than its rest wavelength. Redshift is related to distance for galaxies, but the number in the CSV is a wavelength ratio. Also, astronomers often write redshift as z. z is a brightness input. redshift is the target

fedesoriano (2022), Stellar Classification Dataset - SDSS17, https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17

In [80]:
%pip install kagglehub
%pip install seaborn

import os
import pandas as pd
import numpy as np
import kagglehub
import seaborn as sns
import matplotlib.pyplot as plt

path = kagglehub.dataset_download("fedesoriano/stellar-classification-dataset-sdss17")
stars = pd.read_csv(os.path.join(path, "star_classification.csv"))
print(stars.shape)
stars.head()

(100000, 18)


,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842


In [8]:
stars.describe()

,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,redshift,plate,MJD,fiber_ID
count,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.0,100000.000000,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000
mean,1.237665e+18,177.629117,24.135305,21.980468,20.531387,19.645762,19.084854,18.668810,4481.366060,301.0,3.511610,186.130520,5.783882e+18,0.576661,5137.009660,55588.647500,449.312740
std,8.438560e+12,96.502241,19.644665,31.769291,31.750292,1.854760,1.757895,31.728152,1964.764593,0.0,1.586912,149.011073,3.324016e+18,0.730707,2952.303351,1808.484233,272.498404
min,1.237646e+18,0.005528,-18.785328,-9999.000000,-9999.000000,9.822070,9.469903,-9999.000000,109.000000,301.0,1.000000,11.000000,2.995191e+17,-0.009971,266.000000,51608.000000,1.000000
25%,1.237659e+18,127.518222,5.146771,20.352353,18.965230,18.135828,17.732285,17.460677,3187.000000,301.0,2.000000,82.000000,2.844138e+18,0.054517,2526.000000,54234.000000,221.000000
50%,1.237663e+18,180.900700,23.645922,22.179135,21.099835,20.125290,19.405145,19.004595,4188.000000,301.0,4.000000,146.000000,5.614883e+18,0.424173,4987.000000,55868.500000,433.000000
75%,1.237668e+18,233.895005,39.901550,23.687440,22.123767,21.044785,20.396495,19.921120,5326.000000,301.0,5.000000,241.000000,8.332144e+18,0.704154,7400.250000,56777.000000,645.000000
max,1.237681e+18,359.999810,83.000519,32.781390,31.602240,29.571860,32.141470,29.383740,8162.000000,301.0,6.000000,989.000000,1.412694e+19,7.011245,12547.000000,58932.000000,1000.000000


In [74]:
stars.groupby('class')['redshift'].agg(['mean','std','count','min','max'])

,mean,std,count,min,max
class,,,,,
GALAXY,0.421596,0.264858,59445,-0.009971,1.995524
QSO,1.719676,0.913954,18961,0.000461,7.011245
STAR,-0.000115,0.000465,21594,-0.004136,0.004153


The file mixes galaxies, stars and quasars, but our goal is to predict galaxy redshift. Stars have redshift near 0 and quasars are very far away, so mixing them would confuse the models.

In [45]:
galaxies = stars[stars['class']=='GALAXY'][["u", "g", "r", "i", "z", "redshift"]]
galaxies

,u,g,r,i,z,redshift
0,23.87882,22.27530,20.39501,19.16573,18.79371,0.634794
1,24.77759,22.83188,22.58444,21.16812,21.61427,0.779136
2,25.26307,22.66389,20.60976,19.34857,18.94827,0.644195
3,22.13682,23.77656,21.61162,20.50454,19.25010,0.932346
4,19.43718,17.58028,16.49747,15.97711,15.54461,0.116123
...,...,...,...,...,...,...
99995,22.16759,22.97586,21.90404,21.30548,20.73569,0.000000
99996,22.69118,22.38628,20.45003,19.75759,19.41526,0.404895
99997,21.16916,19.26997,18.20428,17.69034,17.35221,0.143366
99998,25.35039,21.63757,19.91386,19.07254,18.62482,0.455040


In [70]:
print(galaxies.describe().round(4))
print(galaxies.shape)
print("missing data:", galaxies.isnull().sum().sum())
print("Duplicated data:", galaxies.duplicated().sum())

                u           g           r           i           z    redshift
count  59445.0000  59445.0000  59445.0000  59445.0000  59445.0000  59445.0000
mean      22.5874     20.9061     19.5876     18.8520     18.4492      0.4216
std        2.2644      2.1078      1.8741      1.6898      1.6563      0.2649
min       13.8980     12.6790     11.7466     11.2996     10.8974     -0.0100
25%       20.7948     18.9127     17.8172     17.3797     17.0957      0.1645
50%       22.8418     21.5822     20.1046     19.2237     18.7666      0.4563
75%       24.2074     22.4451     20.9746     19.9471     19.4723      0.5947
max       29.3256     31.6022     29.5719     30.1636     29.3837      1.9955
(59445, 6)
missing data: 0
Duplicated data: 0


#### X = the 5 brightness columns (inputs).
#### y = redshift (what to predict).
Split both into 80% train and 20% test, using the same random shuffle for X and y.

In [84]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

X = galaxies[["u", "g", "r", "i", "z"]]
y = galaxies["redshift"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(47556, 5) (11889, 5)


# Classification — Identifying Pulsar Signals
The pulsar_data_train.csv dataset contains radio signals that were flagged as possible pulsars. Pulsars are rotating neutron stars that can produce regular pulses detectable by radio telescopes. Each row contains eight measurements that summarize the shape of a candidate pulse and how clearly it stands out from background noise. The target_class column identifies each candidate as a pulsar (1) or a non-pulsar (0).

Our goal is to use those eight measurements to predict whether a candidate signal comes from a pulsar. Since there are two possible outcomes, this is a binary classification problem. We will handle missing values and compare the models using precision, recall, and F1 score. Most candidates are non-pulsars, so overall accuracy alone could hide how many actual pulsars a model misses.

| Column, in file order | What it tells you |
|---|---|
| Mean of the integrated profile | Average height of the pulse profile. |
| Standard deviation of the integrated profile | How much the profile values vary. |
| Excess kurtosis of the integrated profile | How strongly unusually extreme profile values contribute, relative to a normal distribution. |
| Skewness of the integrated profile | Which side of the distribution of profile values has the longer tail. |
| Mean of the DM-SNR curve | Average signal-to-noise ratio across the trial corrections. |
| Standard deviation of the DM-SNR curve | How much signal-to-noise varies across those trials. |
| Excess kurtosis of the DM-SNR curve | How strongly unusually extreme signal-to-noise values contribute. |
| Skewness of the DM-SNR curve | Which side of the distribution of signal-to-noise values has the longer tail. |
| `target_class` | The answer we want to predict. `1` means pulsar and `0` means non-pulsar. |